# Heat pump performance

In this exercise you will estimate the annual heat load of a building as well as determine how much of the load can be covered by an air-to-air heat pump. The case will be based on Tina's house - so it's a real system!

The first step is to import a few Python pacakges.

In [ ]:
# Install pvlib in Google Colab as this is not a standard package.
!pip install pvlib

In [8]:
import pvlib  # library for retrieving weather data & modeling photovolatics
import pandas as pd  # library for data analysis
import matplotlib.pyplot as plt  # library for plotting
import numpy as np  # library for math and linear algebra

## Step 0: Investigate heat pump datasheet

You can find the datasheet for Tina's heat pump [here](https://heatnow.dk/wp-content/uploads/2021/09/Specification_Sheet_9404.pdf).


Determine the following characteristics:
- What is the maxium and minimum heating capacity?
- What is the maximum and minimum COP?

## Step 1: Define a location

A location is defined by a latitude and longitude according to the convention of [ISO 6709](https://en.wikipedia.org/wiki/ISO_6709). Specifically, latitude is in degrees north of the equator and the longitude is in degrees east of the prime meridian.

The coordinates corresponds to Sisimiut.

In [9]:
latitude = 66.9343
longitude = -53.6748

## Step 2: Retrieve irradiance data from NASA POWER

In this step, weather data is retrieved from the NASA POWER [dataset](https://power.larc.nasa.gov/data-access-viewer/) using the pvlib function [``get_nasa_power``](https://pvlib-python.readthedocs.io/en/latest/reference/generated/pvlib.iotools.get_nasa_power.html).

👉 Simply execute this cell without making any changes.

In [10]:
# Define start and end date
start = "2025-01-01"
end = "2025-12-31"

parameters = parameters=['temp_air']

data, meta = pvlib.iotools.get_nasa_power(
    latitude, longitude, start, end, parameters)

data

,temp_air
2025-01-01 00:00:00+00:00,-13.29
2025-01-01 01:00:00+00:00,-13.19
2025-01-01 02:00:00+00:00,-12.90
2025-01-01 03:00:00+00:00,-12.58
2025-01-01 04:00:00+00:00,-12.22
...,...
2025-12-31 19:00:00+00:00,0.52
2025-12-31 20:00:00+00:00,0.03
2025-12-31 21:00:00+00:00,-0.14
2025-12-31 22:00:00+00:00,-0.06


## Step 3: Plot the temperature

👉 In the code cell below, plot the ambient temperature for the full year.

In [1]:
# Write your code here


## Step 4: Estimate heat load

The ambient temperature is the main driver of heat losses from a house.

We will be simulating Tina's house, which is a single family house which approximately has a heat loss coefficient of 140 W/K. Remember how to estimate this from Martin's lecture?

👉 Calculate the heat load for the year assuming an indoor temperature of 20 °C.

*Hint: remember that you cannot have negative heat load. You can use ``.clip(lower=0)`` to remove negative values.*





In [ ]:
# Write your code here
heatload = 

## Step 5: Heat pump capacity

In this step we'll calculate the maximum heat pump capacity for each time step. The capacity depends on the source temperature (ambient air) and the supply temperature (indoor supply air temperature). We will assume a supply temperature of 20 degrees C (which corresponds to the values given in the datasheet).


👉 Skip the below complicated code cell and move on to the next step.

In [11]:
from __future__ import annotations
import warnings
from collections.abc import Iterable
import pandas as pd

# ---------------------------------------------------------------------------
# Datasheet points  [T_outdoor_db_°C, capacity_kW, COP]
# Source: Panasonic HZ25XKE datasheet, indoor 20°C DB, EN14825
# ---------------------------------------------------------------------------
_DATASHEET: list[tuple[float, float, float]] = [
    (-25.0, 3.60, 2.22),
    (-20.0, 4.20, 2.40),
    (-15.0, 4.78, 2.54),
    ( -7.0, 5.00, 2.58),
    (  7.0, 3.20, 5.61),
]

T_MIN: float = _DATASHEET[0][0]   # -25 °C
T_MAX: float = _DATASHEET[-1][0]  #   7 °C
T_INDOOR_REF: float = 20.0        # °C DB — fixed datasheet indoor condition


def _interp(x: float, x0: float, x1: float, y0: float, y1: float) -> float:
    """Linear interpolation — no external dependencies."""
    return y0 + (y1 - y0) * (x - x0) / (x1 - x0)


def _lookup_scalar(t_outdoor: float) -> tuple[float, float]:
    """
    Return (capacity_kW, COP) for a single outdoor temperature by linear
    interpolation over datasheet points.
    Values outside [T_MIN, T_MAX] are clamped; caller is responsible for
    issuing any warnings before calling this.
    """
    t = max(T_MIN, min(T_MAX, t_outdoor))

    for i in range(len(_DATASHEET) - 1):
        t0, q0, c0 = _DATASHEET[i]
        t1, q1, c1 = _DATASHEET[i + 1]
        if t0 <= t <= t1:
            return _interp(t, t0, t1, q0, q1), _interp(t, t0, t1, c0, c1)

    return _DATASHEET[-1][1], _DATASHEET[-1][2]


def _carnot_cop(t_cond_air: float, t_evap_air: float) -> float:
    """Ideal Carnot COP given condenser-side and evaporator-side air temps."""
    t_cond = t_cond_air + 5.0 + 273.15   # K  (≈ refrigerant condensing temp)
    t_evap = t_evap_air - 8.0 + 273.15   # K  (≈ refrigerant evaporating temp)
    return t_cond / max(t_cond - t_evap, 1.0)


def heating_performance(
    t_outdoor: float | Iterable[float],
    t_indoor: float = T_INDOOR_REF,
) -> pd.DataFrame:
    """
    Estimate heating performance for the Panasonic CU-HZ25XKE.

    Parameters
    ----------
    t_outdoor : float or iterable of float
        Outdoor air dry-bulb temperature(s) in °C.
        Valid datasheet range: -25 to +7 °C. Values outside this range are
        clamped to the nearest endpoint and a warning is issued.
    t_indoor : float, optional
        Indoor air dry-bulb temperature in °C.
        Default is 20°C (the EN14825 datasheet reference condition).
        Deviations are corrected via Carnot lift scaling — use with caution
        beyond ±5°C from the reference.

    Returns
    -------
    pd.DataFrame
        Index  : t_outdoor_c (°C)
        Columns: capacity_kw, cop, input_power_w

    Examples
    --------
    >>> heating_performance(-15)
       capacity_kw   cop  input_power_w
    t_outdoor_c
    -15.0         4.780  2.54        1882.7

    >>> heating_performance([-25, -15, -7, 7])
       capacity_kw   cop  input_power_w
    t_outdoor_c
    -25.0         3.600  2.22        1621.6
    -15.0         4.780  2.54        1882.7
    -7.0          5.000  2.58        1937.9
     7.0          3.200  5.61         570.4
    """
    # Normalise input to a list of floats
    if isinstance(t_outdoor, (int, float)):
        temps = [float(t_outdoor)]
    else:
        temps = [float(t) for t in t_outdoor]

    # Warn once per out-of-range direction
    below = [t for t in temps if t < T_MIN]
    above = [t for t in temps if t > T_MAX]
    if below:
        warnings.warn(
            f"{len(below)} temperature(s) below datasheet minimum ({T_MIN}°C): "
            f"{below}. Clamped to {T_MIN}°C — results may be optimistic.",
            UserWarning, stacklevel=2,
        )
    if above:
        warnings.warn(
            f"{len(above)} temperature(s) above datasheet maximum ({T_MAX}°C): "
            f"{above}. Clamped to {T_MAX}°C.",
            UserWarning, stacklevel=2,
        )

    # Carnot scale factor for indoor temperature deviation
    if t_indoor != T_INDOOR_REF:
        carnot_scale_ref = _carnot_cop(T_INDOOR_REF, 0.0)  # placeholder; computed per-point below
        use_indoor_correction = True
    else:
        use_indoor_correction = False

    rows = []
    for t in temps:
        cap, cop = _lookup_scalar(t)

        if use_indoor_correction:
            scale = _carnot_cop(t_indoor, t) / _carnot_cop(T_INDOOR_REF, t)
            cap *= scale
            cop *= scale

        input_w = (cap / cop) * 1000.0
        rows.append({
            "t_outdoor_c":    t,
            "capacity_kw":    round(cap, 3),
            "cop":            round(cop, 3),
            "input_power_w":  round(input_w, 1),
        })

    df = pd.DataFrame(rows).set_index("t_outdoor_c")
    return df

In [13]:
performance = heating_performance(data["temp_air"], t_indoor=20)

performance

C:\Users\arajen\AppData\Local\Temp\ipykernel_35772\1603332614.py:1: UserWarning: 10 temperature(s) below datasheet minimum (-25.0°C): [-25.18, -25.65, -26.02, -26.17, -26.07, -25.76, -25.34, -25.1, -25.15, -25.11]. Clamped to -25.0°C — results may be optimistic.
  performance = heating_performance(data["temp_air"], t_indoor=20)
C:\Users\arajen\AppData\Local\Temp\ipykernel_35772\1603332614.py:1: UserWarning: 423 temperature(s) above datasheet maximum (7.0°C): [7.49, 8.16, 8.68, 9.06, 9.31, 9.44, 9.38, 9.09, 8.59, 7.77, 7.38, 7.54, 7.51, 7.41, 7.25, 7.64, 7.89, 7.77, 7.54, 7.34, 7.13, 7.03, 7.92, 8.53, 8.85, 8.92, 8.66, 8.22, 7.55, 7.69, 8.44, 8.79, 8.74, 8.36, 7.71, 7.11, 7.06, 7.71, 7.93, 7.8, 7.56, 7.35, 7.12, 7.06, 7.07, 7.1, 7.04, 7.27, 7.39, 7.17, 7.15, 7.39, 7.76, 8.23, 8.25, 7.83, 7.26, 7.35, 7.44, 7.32, 7.18, 7.53, 7.79, 7.81, 7.68, 7.47, 7.15, 7.17, 7.64, 8.15, 8.47, 8.4, 7.96, 7.47, 7.24, 7.16, 7.16, 7.03, 7.68, 7.93, 7.87, 7.7, 7.58, 7.47, 7.4, 7.35, 7.37, 7.36, 7.18, 7.35, 7

,capacity_kw,cop,input_power_w
t_outdoor_c,,,
-13.29,4.827,2.549,1894.0
-13.19,4.830,2.549,1894.7
-12.90,4.838,2.550,1896.8
-12.58,4.847,2.552,1899.0
-12.22,4.856,2.554,1901.6
...,...,...,...
0.52,4.033,4.208,958.6
0.03,4.096,4.101,998.7
-0.14,4.118,4.065,1013.1


## Step 6: Plot the heat capacity, cop, and electricity consumption?

The data is in the ``performance`` dataframe.

In [14]:
# Write your code here

## Step 7: Calculate heat pump heat supply

Now we know the heat load and the maximum heat capacity of the heat pump.

Whenever the heat load is greater than the heat capacity we need to use an alternative heating source (direct electric or oil boiler).

👉Determine what percentage of the heating load can be met by the heat pump?

In [ ]:
# Write your code


## Wrap up

Think about how this study was simplified? What assumptions were made?
